# 🫁 MedVision AI — Google Colab Training Notebook

**Final-Year B.Tech AI/ML Project**

This notebook trains MedVision AI models on a **free GPU** in Google Colab.

## Before you begin
1. **Enable GPU**: Runtime → Change runtime type → Hardware accelerator → **T4 GPU** → Save
2. **Upload dataset**: See Cell 2 for instructions
3. **Run cells in order** from top to bottom

---
> ⚠️ **Medical Disclaimer**: Educational research prototype only. Not for clinical use.

In [ ]:
# Cell 1 — Verify GPU is available
import torch
print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('GPU Memory:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), 'GB')
else:
    print('⚠️ No GPU found! Go to Runtime → Change runtime type → T4 GPU')

## Step 2 — Dataset Setup

**Option A: Kaggle API (Recommended)**
1. Go to https://www.kaggle.com/settings → API → Create New Token → download `kaggle.json`
2. Upload `kaggle.json` to Colab (click the Files icon on the left)
3. Run the cell below

**Option B: Manual Upload**
1. Download from https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia
2. Upload the ZIP to Google Drive and mount it (see Option B cell)

In [ ]:
# Cell 2A — Download via Kaggle API
import os
from pathlib import Path

# Setup Kaggle credentials
os.makedirs('/root/.kaggle', exist_ok=True)
# Upload kaggle.json first, then run this cell
if Path('/content/kaggle.json').exists():
    os.system('cp /content/kaggle.json /root/.kaggle/')
    os.system('chmod 600 /root/.kaggle/kaggle.json')
    os.system('pip install kaggle -q')
    os.system('kaggle datasets download -d paultimothymooney/chest-xray-pneumonia -p /content/')
    os.system('unzip -q /content/chest-xray-pneumonia.zip -d /content/')
    print('Dataset downloaded!')
    # Find the actual chest_xray folder
    for root, dirs, files in os.walk('/content'):
        if 'train' in dirs and 'test' in dirs:
            print(f'Dataset root: {root}')
            DATASET_ROOT = root
            break
else:
    print('kaggle.json not found. Please upload it first.')

In [ ]:
# Cell 2B — Mount Google Drive (Alternative if you uploaded ZIP to Drive)
# from google.colab import drive
# drive.mount('/content/drive')
# 
# import zipfile
# ZIP_PATH = '/content/drive/MyDrive/chest-xray-pneumonia.zip'  # adjust path
# with zipfile.ZipFile(ZIP_PATH, 'r') as z:
#     z.extractall('/content/')
# DATASET_ROOT = '/content/chest_xray'
# print('Dataset extracted!')

In [ ]:
# Cell 3 — Install dependencies
!pip install grad-cam seaborn scikit-learn tqdm python-dotenv PyYAML -q
print('Dependencies installed!')

In [ ]:
# Cell 4 — Clone/upload project code
# Option A: Clone from GitHub (if you've pushed the code)
# !git clone https://github.com/YOUR_USERNAME/MedVision-AI.git /content/MedVision-AI

# Option B: Upload project zip from local machine
# from google.colab import files
# uploaded = files.upload()  # select the MedVision-AI.zip
# !unzip MedVision-AI.zip -d /content/

# For now, we'll set up the src path manually
import sys
PROJECT_DIR = '/content/MedVision-AI'  # adjust if your path differs
sys.path.insert(0, PROJECT_DIR)
import os
os.chdir(PROJECT_DIR)
print(f'Working directory: {os.getcwd()}')

In [ ]:
# Cell 5 — Set dataset path and load config
import os
os.environ['DATASET_ROOT'] = DATASET_ROOT  # set from Cell 2A or 2B
os.environ['DEVICE'] = 'cuda'

from src.utils.config_loader import get_config
cfg = get_config()
print('Config loaded!')
print(f'Dataset root: {cfg["dataset"]["root"]}')

In [ ]:
# Cell 6 — Phase 2: Dataset Exploration
from pathlib import Path
from src.data.dataset import (
    find_dataset_root, print_dataset_summary,
    count_images, detect_corrupt_images, compute_class_weights
)
from src.utils.visualization import plot_class_distribution, plot_sample_images

dataset_root = find_dataset_root(cfg)
print_dataset_summary(dataset_root, cfg)

# Class distribution charts
for split in ['train', 'test']:
    counts = count_images(dataset_root, split)
    if counts:
        plot_class_distribution(counts, split_name=split.capitalize(),
                                save_dir='reports')

# Show sample images (Colab inline)
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('inline')  # use inline backend for Colab
from PIL import Image
import numpy as np

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for row, cls in enumerate(['NORMAL', 'PNEUMONIA']):
    cls_dir = dataset_root / 'train' / cls
    paths = list(cls_dir.glob('*.jpeg'))[:4] + list(cls_dir.glob('*.jpg'))[:4]
    for col, p in enumerate(paths[:4]):
        axes[row][col].imshow(Image.open(p), cmap='gray')
        axes[row][col].set_title(cls, color='green' if cls=='NORMAL' else 'red')
        axes[row][col].axis('off')
plt.suptitle('Sample Chest X-Ray Images', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 7 — Phase 3: Build DataLoaders
from src.data.preprocessing import get_train_transform, get_val_transform
from src.data.dataset import build_dataloaders, compute_class_weights

image_size   = cfg['preprocessing']['image_size']
train_tf     = get_train_transform(image_size)
val_tf       = get_val_transform(image_size)

train_loader, val_loader, test_loader, class_names = build_dataloaders(
    cfg=cfg,
    train_transform=train_tf,
    val_transform=val_tf,
    dataset_root=dataset_root,
)
class_weights = compute_class_weights(dataset_root, cfg)
print(f'Class names: {class_names}')
print(f'Class weights: {class_weights}')

In [ ]:
# Cell 8 — Phase 4: Train Baseline CNN
from src.models.cnn import build_baseline_cnn
from src.training.train import train_model
from src.utils.visualization import plot_training_curves

cnn_model = build_baseline_cnn(cfg)

cnn_history = train_model(
    model=cnn_model,
    train_loader=train_loader,
    val_loader=val_loader,
    cfg=cfg,
    model_type='cnn',
    class_weights=class_weights,
    save_name='best_cnn.pth',
)

plot_training_curves(cnn_history, model_name='Baseline CNN', save_dir='reports')

In [ ]:
# Cell 9 — Evaluate CNN on Test Set
from src.training.evaluate import evaluate_model

cnn_metrics = evaluate_model(
    model=cnn_model,
    test_loader=test_loader,
    cfg=cfg,
    class_names=class_names,
    model_name='Baseline CNN',
)

In [ ]:
# Cell 10 — Phase 5A: ResNet-18 Frozen Backbone
from src.models.resnet import build_resnet

resnet_model = build_resnet(cfg)  # backbone frozen

# Train only the head for 10 epochs
head_cfg = dict(cfg)
head_cfg['resnet'] = dict(cfg['resnet'])
head_cfg['resnet']['epochs'] = 10

history_a = train_model(
    model=resnet_model,
    train_loader=train_loader,
    val_loader=val_loader,
    cfg=head_cfg,
    model_type='resnet',
    class_weights=class_weights,
    save_name='best_resnet18_head.pth',
)

In [ ]:
# Cell 11 — Phase 5B: Fine-Tuning (Unfreeze last 2 layer groups)
resnet_model.unfreeze_layers(2)

history_b = train_model(
    model=resnet_model,
    train_loader=train_loader,
    val_loader=val_loader,
    cfg=cfg,
    model_type='resnet',
    class_weights=class_weights,
    save_name='best_resnet18.pth',
)

combined = {
    'train_loss': history_a['train_loss'] + history_b['train_loss'],
    'val_loss':   history_a['val_loss']   + history_b['val_loss'],
    'train_acc':  history_a['train_acc']  + history_b['train_acc'],
    'val_acc':    history_a['val_acc']    + history_b['val_acc'],
}
plot_training_curves(combined, model_name='ResNet-18', save_dir='reports')

In [ ]:
# Cell 12 — Evaluate ResNet-18 on Test Set
resnet_metrics = evaluate_model(
    model=resnet_model,
    test_loader=test_loader,
    cfg=cfg,
    class_names=class_names,
    model_name='ResNet-18',
)

In [ ]:
# Cell 13 — Grad-CAM Demo on a Test Image
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('inline')
from PIL import Image
import numpy as np
from src.inference.predict import predict_single_image
from src.utils.visualization import overlay_gradcam

# Pick a PNEUMONIA test image
test_pneumonia_dir = dataset_root / 'test' / 'PNEUMONIA'
sample_img_path = list(test_pneumonia_dir.glob('*.jpeg'))[0]
print(f'Using: {sample_img_path.name}')

result = predict_single_image(
    image=sample_img_path,
    model=resnet_model,
    cfg=cfg,
    generate_cam=True,
)

print(f"Predicted: {result['predicted_class']}")
print(f"Confidence: {result['confidence']*100:.1f}%")
print(f"Class scores: {result['class_scores']}")

if result['heatmap'] is not None:
    blended = overlay_gradcam(
        result['original_image'], result['heatmap'], alpha=0.4
    )
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].imshow(result['original_image'])
    axes[0].set_title('Original X-Ray', fontsize=12)
    axes[0].axis('off')
    axes[1].imshow(blended)
    axes[1].set_title(f'Grad-CAM: {result["predicted_class"]} ({result["confidence"]*100:.1f}%)',
                      fontsize=12)
    axes[1].axis('off')
    plt.suptitle('Grad-CAM Explainability', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

In [ ]:
# Cell 14 — Download trained weights to your local machine
from google.colab import files
import os

weights_dir = 'artifacts/checkpoints'
for fname in ['best_cnn.pth', 'best_resnet18.pth']:
    fpath = os.path.join(weights_dir, fname)
    if os.path.exists(fpath):
        print(f'Downloading {fname} ...')
        files.download(fpath)
    else:
        print(f'{fname} not found — check training completed successfully.')

print('\nInstructions:')
print('Place downloaded .pth files in MedVision-AI/artifacts/checkpoints/')
print('Then run: streamlit run app.py')